# Launch-in-a-Day: build a team of AI agents with Google ADK + A2A

In this notebook we build up the Launch-in-a-Day team one idea at a time:

1. **An agent** = an LLM + instructions
2. **A tool** = a Python function the agent can ask us to run
3. **A2A** = give the agent an address and a public "business card"
4. **A Director** that calls a remote agent it has never seen the code of
5. **The real team**: approval gate and safety filter
6. **A full launch**: idea → brand → animated page → you type *yes* → live on GitHub Pages

> **MCP gives an agent hands (tools). A2A gives an agent colleagues (other agents).**

### Before you start
- Finish the setup in `README.md`: `.venv` created, `uv pip install -r requirements.txt`, and a `.env`
  with `GOOGLE_API_KEY`, `MODEL` and `BUILDER_MODEL`.
- Install Jupyter in the same environment and open this notebook **from the `launch-in-a-day` folder**:
  ```bash
  uv pip install jupyterlab
  jupyter lab workshop.ipynb
  ```
  In VS Code instead: open `workshop.ipynb`, click **Select Kernel** and pick `.venv`.
- Run the cells **from top to bottom**. Each cell says how many model calls it makes.
- The free tier allows about **15 requests per minute** per model. If a cell says the limit was reached,
  wait a minute and run that cell again.

## Part 0 · Setup
Load the settings from `.env` (the key is never printed) and make our agent folders importable.

In [1]:
import os, sys, re, json, socket, threading, time
from pathlib import Path

PROJECT = Path.cwd()  # this notebook lives in the launch-in-a-day folder
if not (PROJECT / "brand_agent").exists():
    raise RuntimeError("Open this notebook from the launch-in-a-day folder.")
sys.path.insert(0, str(PROJECT))  # so we can import brand_agent, builder_agent, ...

from dotenv import load_dotenv
load_dotenv(PROJECT / ".env")  # GOOGLE_API_KEY, MODEL, BUILDER_MODEL

MODEL = os.environ["MODEL"]
print("API key loaded:", bool(os.getenv("GOOGLE_API_KEY")))
print("MODEL         =", MODEL)
print("BUILDER_MODEL =", os.getenv("BUILDER_MODEL"))

API key loaded: True
MODEL         = gemini-3.5-flash-lite
BUILDER_MODEL = gemini-4-hydrogen


Two small helpers keep the rest of the notebook short. They are **not part of the product**:
- `Chat(agent)` talks to an agent and prints what happens: 🔧 tool calls, 📦 tool results, 🤖 answers.
- `start_server(app, port)` runs an A2A agent in the background, like `uvicorn ... --port` in a Terminal.

In [2]:
import logging
import httpx, uvicorn
from google.adk.runners import InMemoryRunner
from google.genai import types
from IPython.display import HTML, display

# ADK prints long tracebacks for model errors; Chat explains them in one line instead.
logging.getLogger("google_adk").setLevel(logging.CRITICAL)


def short(value, limit=300):
    """Shortens long results (like SVG code) so the output stays readable."""
    text = re.sub(r"```svg.*?```", "```svg …```", str(value), flags=re.S)
    text = re.sub(r"<svg.*?</svg>", "<svg …>", text, flags=re.S)
    return text if len(text) <= limit else text[:limit] + " …"


class Chat:
    """A conversation with one agent. It keeps the session, so the agent remembers earlier turns."""

    def __init__(self, agent):
        self.runner = InMemoryRunner(agent=agent, app_name="workshop")
        self.session = None

    async def send(self, text):
        if self.session is None:
            self.session = await self.runner.session_service.create_session(app_name="workshop", user_id="me")
        print(f"🙂 you: {text}")
        message = types.Content(role="user", parts=[types.Part(text=text)])
        answer = ""
        try:
            async for event in self.runner.run_async(user_id="me", session_id=self.session.id, new_message=message):
                for part in (event.content.parts if event.content else []):
                    if part.function_call:
                        print(f"🔧 {event.author} calls {part.function_call.name}({short(part.function_call.args, 150)})")
                    elif part.function_response:
                        print(f"📦 {part.function_response.name} returned: {short(part.function_response.response)}")
                    elif part.text and not part.thought:
                        answer = part.text
                        print(f"🤖 {event.author}: {short(part.text, 1500)}")
        except Exception as error:
            if "RESOURCE_EXHAUSTED" in str(error) or "429" in str(error):
                print("⏳ Free-tier limit reached (about 15 requests per minute). Wait a minute and run this cell again.")
            elif "UNAVAILABLE" in str(error) or "503" in str(error):
                print("⏳ The model is busy right now (high demand). Run this cell again in a minute.")
            else:
                raise
        return answer


def start_server(app, port):
    """Runs an A2A web app in the background, like `uvicorn ... --port <port>` in a Terminal."""
    with socket.socket() as s:
        if s.connect_ex(("localhost", port)) == 0:
            print(f"Something already runs on port {port} (maybe in your Terminal): using it.")
            return
    server = uvicorn.Server(uvicorn.Config(app, host="localhost", port=port, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    while not server.started:
        time.sleep(0.1)
    print(f"Serving on http://localhost:{port}")


print("Helpers ready.")

Helpers ready.


## Part 1 · An agent is an LLM with a job description
Think of a **new employee**:
- `model` is the brain you hire.
- `instruction` is the job description the employee reads: written **for the agent itself**.
- `description` is their line in the company directory: written **for other agents**, so they know what to ask it for.

*Cost: 1 model call.*

In [3]:
from google.adk.agents import Agent

tagline_agent = Agent(
    name="tagline_agent",                                   # a short ID, no spaces
    model=MODEL,                                            # the brain (from .env)
    description="Writes one catchy tagline for a startup idea.",  # for OTHER agents
    instruction=(                                           # for ITSELF
        "Given a startup idea, reply with ONE catchy tagline of at most 10 words. Nothing else."
    ),
)

await Chat(tagline_agent).send("a dating app for houseplants")

🙂 you: a dating app for houseplants
🤖 tagline_agent: Swipe right for your soulmate stem.


'Swipe right for your soulmate stem.'

🎤 *"This is a chatbot with a job description."*

## Part 2 · Give it a tool
A **tool** is a plain Python function. **The model never runs code**: it asks ADK to run the function,
then reads the result. Our Brand agent has one tool, `make_logo_svg`, that draws the logo.

First, the tool on its own, with no AI involved. *Cost: 0 model calls.*

In [4]:
from brand_agent.agent import make_logo_svg

logo = make_logo_svg("Leafy Match", "#1B4D3E", "#2ECC71")  # plain Python
display(HTML(logo["svg"]))
print(logo["svg"][:120], "…")

<svg xmlns="http://www.w3.org/2000/svg" width="120" height="120" viewBox="0 0 120 120"><defs><linearGradient id="g1B4D3E …


How does the model know the tool exists? ADK turns the function's **name, docstring and type hints**
into a short label that is sent to Gemini. This is everything the model ever sees about the tool.
*Cost: 0 model calls.*

In [5]:
from google.adk.tools import FunctionTool

label = FunctionTool(make_logo_svg)._get_declaration()  # what ADK sends to Gemini
print("name:       ", label.name)
print("description:", label.description)
print("parameters: ", json.dumps(label.parameters_json_schema["properties"], indent=2))

name:        make_logo_svg
description: Draws a polished logo badge: the company's first letter on a gradient tile.

Args:
    name: The company name, e.g. "Leafy Match".
    color: The dark brand color as a hex code, e.g. "#1B4D3E".
    accent_color: The bright brand color as a hex code, e.g. "#2ECC71".

Returns:
    A dict with key "svg": the logo as SVG code.
parameters:  {
  "name": {
    "title": "Name",
    "type": "string"
  },
  "color": {
    "title": "Color",
    "type": "string"
  },
  "accent_color": {
    "title": "Accent Color",
    "type": "string"
  }
}


Now the real Brand agent from `brand_agent/agent.py`. Read its instruction, then ask it for a brand.
Watch the output: the model **decides** to call the tool 🔧, ADK runs it 📦, then the model answers 🤖.

*Cost: 2 model calls (decide to call the tool, then answer).*

In [6]:
from brand_agent.agent import root_agent as brand_agent

print(brand_agent.instruction)

You are a brand designer. Given a startup idea:
1. Invent a catchy company name, a one-sentence tagline, 3 hex colors (one dark, one bright, one light), a design vibe and one emoji that fits the idea.
2. Call make_logo_svg with the name, the dark color and the bright color.
3. Reply with exactly these lines and nothing else:
Name: <name>
Tagline: <tagline>
Colors: <dark> <bright> <light>
Vibe: <exactly one of: Playful Pop, Calm Organic, Bold Tech, Luxury Minimal>
Emoji: <one emoji>
Logo:
```svg
<the svg from the tool, unchanged>
```


In [7]:
answer = await Chat(brand_agent).send("a dating app for houseplants")

svg = re.search(r"<svg.*?</svg>", answer, re.S)
if svg:
    display(HTML(svg.group(0)))

🙂 you: a dating app for houseplants
🔧 brand_agent calls make_logo_svg({'color': '#1A3A2B', 'name': 'SproutMate', 'accent_color': '#FF6B8B'})
📦 make_logo_svg returned: {'svg': '<svg …>'}
🤖 brand_agent: Name: SproutMate
Tagline: Swipe right for root romance and plant your next great love story.
Colors: #1A3A2B #FF6B8B #F4F9F5
Vibe: Playful Pop
Emoji: 🌱
Logo:
```svg …```


🎤 *"The model doesn't run code. It asks us to run a function and reads the result."*

## Part 3 · A2A: an address and a business card
**A2A** turns an agent into a small web service with a public **Agent Card** at
`/.well-known/agent-card.json`. Other agents read the card to learn what it can do.
The card shows **what** the agent does, never **how**: its instruction (the prompt) stays private.

This is the whole of `brand_agent/a2a_server.py`. The key line is `to_a2a(...)`:

In [8]:
print((PROJECT / "brand_agent" / "a2a_server.py").read_text())

"""Expose the Brand agent over A2A on port 8001.

The agent itself (agent.py) is unchanged. This file only gives it an address
and a public "business card" (the Agent Card).

Run from the launch-in-a-day folder:
    uvicorn brand_agent.a2a_server:a2a_app --host localhost --port 8001
Then open:
    http://localhost:8001/.well-known/agent-card.json
"""

from google.adk.a2a.utils.agent_to_a2a import to_a2a

from .agent import root_agent

# Wrap the agent in a small web app. The port here is only written into the
# Agent Card; it must match the --port you give uvicorn.
a2a_app = to_a2a(root_agent, port=8001)



Start it on port 8001. If you already run the Brand agent in a Terminal, the notebook simply uses that one.
*Cost: 0 model calls.*

In [9]:
from brand_agent.a2a_server import a2a_app as brand_app

start_server(brand_app, 8001)

Something already runs on port 8001 (maybe in your Terminal): using it.


Read its business card, exactly like another agent would. Find the `description`, the `skills`
(made from the agent and its tool's docstring) and the address in `supportedInterfaces`.
*Cost: 0 model calls.*

In [ ]:
card = httpx.get("http://localhost:8001/.well-known/agent-card.json").json()
print(json.dumps(card, indent=2, ensure_ascii=False))

🎤 *"Same agent, now it has an address and a business card. Anyone can discover it."*

### X-ray: what A2A looks like on the wire
A2A is just JSON over HTTP. Here is a complete A2A 1.0 request from `xray/send_message.json`.
Note the header `A2A-Version: 1.0`: without it the server assumes the old 0.3 protocol.
The answer is a **task** with an ID, a state and the result.

*Cost: 2 model calls (the Brand agent does its work).*

In [ ]:
request = json.loads((PROJECT / "xray" / "send_message.json").read_text())
print(json.dumps(request, indent=2))

reply = httpx.post("http://localhost:8001/", json=request,
                   headers={"A2A-Version": "1.0"}, timeout=180).json()

if "result" not in reply:
    print("\nA2A error:", reply.get("error"))
else:
    task = reply["result"]["task"]
    texts = [p["text"] for a in task.get("artifacts", []) for p in a["parts"] if "text" in p]
    print("\ntask id :", task["id"])
    print("state   :", task["status"]["state"])
    print("result  :", short("\n".join(texts) or str(task["status"].get("message", ""))))

## Part 4 · A Director that calls a remote agent
The Director is a **project manager with a phone**. It never sees the Brand agent's code or prompt:
only its address and its description.
- `RemoteA2aAgent` is the contact card: a stand-in for an agent that lives somewhere else.
- `AgentTool` lets the Director call the teammate **like a function and get the answer back**,
  so the Director stays in charge (instead of handing the whole conversation over).

*Cost: about 4 model calls (2 Director + 2 Brand).*

In [ ]:
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent
from google.adk.tools.agent_tool import AgentTool

remote_brand = RemoteA2aAgent(
    name="brand_agent",
    agent_card="http://localhost:8001/.well-known/agent-card.json",  # its address
    description="Creates a brand for a startup idea: name, tagline, colors, vibe, emoji and logo.",
)

mini_director = Agent(
    name="mini_director",
    model=MODEL,
    instruction="When the user gives a startup idea, call brand_agent with it, then show its answer.",
    tools=[AgentTool(agent=remote_brand)],
)

await Chat(mini_director).send("a pizza delivery service by drone")

🎤 *"The Director has never seen the Brand agent's code or prompt. It only read its business card."*

Adding a teammate is the same three lines again: that is how the Builder and the Publisher join.

## Part 5 · The real Launch Director
`launch_director/agent.py` has three teammates and two pieces of plain code around the model.
*Cost: 0 model calls in this part.*

In [ ]:
from launch_director.agent import root_agent as director

for tool in director.tools:
    print(f"👥 {tool.name}: {tool.description}")
print()
print(director.instruction)

**The approval gate.** Before *every* tool call, this code runs. If the tool is the Publisher and your
last message is not exactly *yes*, the call is blocked, whatever the model thinks.
It also attaches the saved files, so the model never has to copy a whole website.

In [ ]:
import inspect
from launch_director.agent import approval_gate

print(inspect.getsource(approval_gate))

**The safety filter.** The page will be public, so before saving, code removes scripts, event handlers
and external links from whatever the model wrote.

In [ ]:
from launch_director.agent import make_safe

evil_html = ('<h1 onclick="steal()">Hi</h1><script>alert("hacked")</script>'
             '<img src="https://tracker.example/pixel.gif">')
safe_html, safe_css = make_safe(evil_html, "body { background: url(https://tracker.example/bg.png) }")

print("before:", evil_html)
print("after: ", safe_html)
print("css:   ", safe_css.strip())

🎤 *"The AI proposes, the human approves, and only one agent holds the keys."*

## Part 6 · A full launch
The Director needs all three teammates running:
- **Brand** (port 8001): already started in Part 3.
- **Builder** (port 8002): the next cell starts it. It uses `BUILDER_MODEL`, the strongest free model,
  which can take from 20 seconds to a few minutes when it is busy.
- **Publisher** (port 8003): start it in **its own Terminal**, because it holds the GitHub token and
  that secret should live in its own process only:
  ```bash
  uvicorn publisher_agent.a2a_server:a2a_app --host localhost --port 8003
  ```
*Cost: 0 model calls.*

In [ ]:
from builder_agent.a2a_server import a2a_app as builder_app

start_server(builder_app, 8002)

try:
    publisher_card = httpx.get("http://localhost:8003/.well-known/agent-card.json", timeout=3).json()
    print("Publisher is running:", publisher_card["name"])
except httpx.HTTPError:
    print("Publisher is NOT running yet. Start it in its own Terminal (see above) before Part 6's last cells.")

Give the Director an idea. Watch it call `brand_agent`, then `builder_agent` (the slow, creative step),
save the files to `out/` and ask you to type *yes*.

*Cost: about 6 model calls (3 Director + 2 Brand + 1 Builder).*

In [ ]:
launch = Chat(director)
await launch.send("a dating app for houseplants")

Open the preview in your browser. Wait two seconds for the animations, then scroll down.

In [ ]:
import webbrowser

page = (PROJECT / "out" / "latest" / "index.html").resolve()
print(page)
webbrowser.open(page.as_uri())

**Test the gate.** Anything except *yes* must not publish. The Director either asks you to type *yes*,
or tries to call the Publisher and gets `"status": "blocked"` back from the code.
*Cost: 1-2 model calls.*

In [ ]:
await launch.send("publish it")

### ⚠️ Publish (irreversible and public)
Run the next cell **only when you want this page on GitHub**. It needs the Publisher running in its own
Terminal. You get a live link: GitHub Pages needs about a minute before it works.

*Cost: about 4 model calls (2 Director + 2 Publisher).*

In [ ]:
await launch.send("yes")

## Recap
1. **An agent** = LLM + instructions + tools, running in a loop.
2. **ADK** is the toolkit for building agents and teams of agents.
3. **A2A** lets agents in different places discover each other through an Agent Card and talk in plain JSON,
   without seeing each other's insides.

**MCP gives an agent hands. A2A gives an agent colleagues.**